# ML-KEM-768 KAT Notebook

Run NIST KAT vectors directly on KR260 hardware using the same driver logic as `ml_kem_kat_test.py`.

Recommended flow:
1. Run with `n_vectors = 1`.
2. Increase to `n_vectors = 100`.
3. Optionally run full KAT file.

In [ ]:
import os
import time

from ml_kem_driver import MLKem768, cycles_to_us


def parse_kat_file(path):
    vectors = []
    current = {}
    with open(path, "r") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                if current:
                    vectors.append(current)
                    current = {}
                continue
            if "=" not in line:
                continue
            key, _, val = line.partition("=")
            key = key.strip()
            val = val.strip()
            try:
                current[key] = bytes.fromhex(val)
            except ValueError:
                current[key] = val
    if current:
        vectors.append(current)
    return vectors


In [ ]:
def run_kat(vectors, kem, stop_on_fail=False):
    n = len(vectors)
    pass_ct = 0
    fail_ct = 0
    first_fail = None
    totals = {"keygen": 0, "encaps": 0, "decaps": 0}
    fail_reasons = []

    t0 = time.monotonic()
    for i, v in enumerate(vectors):
        try:
            d = v["d"]
            z = v["z"]
            m = v["m"]
            pk_exp = v["pk"]
            sk_exp = v["sk"]
            ct_exp = v["ct"]
            ss_exp = v["ss"]
        except KeyError as e:
            fail_ct += 1
            fail_reasons.append((i, f"missing KAT field {e}"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        pk, sk, c_kg = kem.keygen(d, z)
        totals["keygen"] += c_kg
        if pk != pk_exp:
            fail_ct += 1
            fail_reasons.append((i, "pk mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue
        if sk != sk_exp:
            fail_ct += 1
            fail_reasons.append((i, "sk mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        ct, ss_enc, c_enc = kem.encaps(pk, m)
        totals["encaps"] += c_enc
        if ct != ct_exp:
            fail_ct += 1
            fail_reasons.append((i, "ct mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue
        if ss_enc != ss_exp:
            fail_ct += 1
            fail_reasons.append((i, "encaps ss mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        ss_dec, c_dec = kem.decaps(sk, ct)
        totals["decaps"] += c_dec
        if ss_dec != ss_exp:
            fail_ct += 1
            fail_reasons.append((i, "decaps ss mismatch"))
            if first_fail is None:
                first_fail = i
            if stop_on_fail:
                break
            continue

        pass_ct += 1
        if (i + 1) % 10 == 0 or (i + 1) == n:
            print(f"[{i+1:4d}/{n}] pass (KG={c_kg}, Enc={c_enc}, Dec={c_dec})")

    wall = time.monotonic() - t0
    return {
        "n": n,
        "pass": pass_ct,
        "fail": fail_ct,
        "first_fail": first_fail,
        "reasons": fail_reasons,
        "totals": totals,
        "wall_seconds": wall,
    }


In [ ]:
script_dir = os.getcwd()
BITSTREAM_DIR = "/root/jupyter_notebooks/verilog_ML_KEM/bitstream"

bitfile = os.environ.get("ML_KEM_BIT", os.path.join(BITSTREAM_DIR, "ml_kem_bd.bit"))
kat_path = os.environ.get("KAT_FILE", os.path.join(script_dir, "KAT_768.txt"))
n_vectors = 1  # change to 100 or None for full file

print(f"bitfile: {bitfile}")
print(f"kat_path: {kat_path}")
print(f"n_vectors: {n_vectors}")
assert os.path.exists(bitfile), f"bitfile not found on board: {bitfile}"
assert os.path.exists(bitfile.replace('.bit', '.hwh')), "missing .hwh next to .bit"
assert os.path.exists(kat_path), f"KAT file not found: {kat_path}"


In [ ]:
import hashlib, os
bit = "/root/jupyter_notebooks/verilog_ML_KEM/bitstream/ml_kem_bd.bit"
print("size:", os.path.getsize(bit))
print("mtime:", os.path.getmtime(bit))
print("md5:", hashlib.md5(open(bit, 'rb').read()).hexdigest())

In [ ]:
# Debug vec #0 — diagnose a KAT regression by locating the first byte diff per stage.
# ML-KEM-768 ct layout: c1 = Compress_10(u), 960 bytes || c2 = Compress_4(v), 128 bytes.
# Run this cell BEFORE the big KAT sweep below when something fails.

def _first_diff(a, b):
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return i
    return None if len(a) == len(b) else min(len(a), len(b))


v0 = parse_kat_file(kat_path)[0]
print(f"KAT vec #0 sizes: d={len(v0['d'])} z={len(v0['z'])} m={len(v0['m'])} "
      f"pk={len(v0['pk'])} sk={len(v0['sk'])} ct={len(v0['ct'])} ss={len(v0['ss'])}")

with MLKem768(bitfile) as kem:
    pk, sk, c_kg = kem.keygen(v0["d"], v0["z"])
    pk_diff = _first_diff(pk, v0["pk"])
    sk_diff = _first_diff(sk, v0["sk"])
    print(f"\nKeyGen ({c_kg} cyc)")
    print(f"  pk match: {pk == v0['pk']}   first diff byte: {pk_diff}")
    print(f"  sk match: {sk == v0['sk']}   first diff byte: {sk_diff}")

    ct, ss_enc, c_enc = kem.encaps(pk, v0["m"])
    ct_diff = _first_diff(ct, v0["ct"])
    ss_diff = _first_diff(ss_enc, v0["ss"])
    region = ""
    if ct_diff is not None:
        region = "(c1 region)" if ct_diff < 960 else f"(c2 region, byte-in-c2={ct_diff - 960})"
    print(f"\nEncaps ({c_enc} cyc)")
    print(f"  ct match: {ct == v0['ct']}   first diff byte: {ct_diff}   {region}")
    print(f"  ss match: {ss_enc == v0['ss']}   first diff byte: {ss_diff}")
    print(f"  ct[0:32]    got: {ct[:32].hex()}")
    print(f"  ct[0:32]    exp: {v0['ct'][:32].hex()}")
    print(f"  ct[960:992] got: {ct[960:992].hex()}   (c2 head)")
    print(f"  ct[960:992] exp: {v0['ct'][960:992].hex()}")
    print(f"  ss got: {ss_enc.hex()}")
    print(f"  ss exp: {v0['ss'].hex()}")

    ss_dec, c_dec = kem.decaps(sk, v0["ct"])   # feed KAT ct, not our (possibly bad) ct
    print(f"\nDecaps ({c_dec} cyc, fed KAT ct)")
    print(f"  ss match: {ss_dec == v0['ss']}   first diff byte: {_first_diff(ss_dec, v0['ss'])}")


In [ ]:
vectors = parse_kat_file(kat_path)
if n_vectors is not None:
    vectors = vectors[:n_vectors]

with MLKem768(bitfile) as kem:
    result = run_kat(vectors, kem)

print("\n===== Summary =====")
print(f"PASS: {result['pass']} / {result['n']}")
print(f"FAIL: {result['fail']} / {result['n']}")

if result["fail"]:
    print(f"First failure index: {result['first_fail']}")
    for idx, reason in result["reasons"][:10]:
        print(f"  vec #{idx}: {reason}")

n_ok = result["pass"]
if n_ok > 0:
    t = result["totals"]
    avg_kg = t["keygen"] / n_ok
    avg_enc = t["encaps"] / n_ok
    avg_dec = t["decaps"] / n_ok
    avg_full = (t["keygen"] + t["encaps"] + t["decaps"]) / n_ok
    print("\nAverage hardware cycles (passed vectors):")
    print(f"  KeyGen: {avg_kg:.1f} cyc ({cycles_to_us(avg_kg):.1f} us)")
    print(f"  Encaps: {avg_enc:.1f} cyc ({cycles_to_us(avg_enc):.1f} us)")
    print(f"  Decaps: {avg_dec:.1f} cyc ({cycles_to_us(avg_dec):.1f} us)")
    print(f"  Full  : {avg_full:.1f} cyc ({cycles_to_us(avg_full):.1f} us)")

print(f"\nWall time: {result['wall_seconds']:.2f} s")
print(f"Per vector wall: {(result['wall_seconds'] / max(1, result['n'])) * 1000:.2f} ms")

assert result["fail"] == 0, "KAT regression failed"
